# 13. (coding) transformer


## What Changed

1. Implemented `softmax` with numerical stability (`x - max(x)`).
2. Implemented `layer_norm` using per-token mean and variance over the last dimension.
3. Implemented self-attention projections (`Q`, `K`, `V`) and scaled dot-product scores.
4. Completed the FFN residual block in `forward` with pre-layernorm.

In [2]:
import numpy as np

def softmax(x, axis=-1):
    x_shifted = x - np.max(x, axis=axis, keepdims=True)
    exp_x = np.exp(x_shifted)
    return exp_x / np.sum(exp_x, axis=axis, keepdims=True)

def layer_norm(x, eps=1e-5):
    mu = np.mean(x, axis=-1, keepdims=True)
    var = np.var(x, axis=-1, keepdims=True)
    return (x - mu) / np.sqrt(var + eps)

def positional_encoding(T, D):
    pe = np.zeros((T, D), dtype=np.float64)
    pos = np.arange(T)[:, None]
    i = np.arange(D)[None, :]
    angle_rates = 1.0 / np.power(10000.0, (2 * (i // 2)) / D)
    angles = pos * angle_rates
    pe[:, 0::2] = np.sin(angles[:, 0::2])
    pe[:, 1::2] = np.cos(angles[:, 1::2])
    return pe

class EncoderLayer:
    def __init__(self, vocab_size, d_model=32, d_ff=64, seed=0):
        rng = np.random.default_rng(seed)
        self.vocab_size = vocab_size
        self.D = d_model
        self.d_ff = d_ff
        self.E = rng.normal(0, 0.02, size=(vocab_size, d_model))
        self.Wq = rng.normal(0, 0.02, size=(d_model, d_model))
        self.Wk = rng.normal(0, 0.02, size=(d_model, d_model))
        self.Wv = rng.normal(0, 0.02, size=(d_model, d_model))
        self.Wo = rng.normal(0, 0.02, size=(d_model, d_model))
        self.W1 = rng.normal(0, 0.02, size=(d_model, d_ff))
        self.b1 = np.zeros((d_ff,))
        self.W2 = rng.normal(0, 0.02, size=(d_ff, d_model))
        self.b2 = np.zeros((d_model,))

    def self_attention(self, x, attn_mask=None):
        B, T, D = x.shape
        scale = 1.0 / np.sqrt(D)
        Q = x @ self.Wq
        K = x @ self.Wk
        V = x @ self.Wv
        scores = (Q @ np.transpose(K, (0, 2, 1))) * scale
        if attn_mask is not None:
            scores = scores + attn_mask[None, :, :]
        A = softmax(scores, axis=-1)
        C = A @ V
        out = C @ self.Wo
        return out, A

    def feed_forward(self, x):
        h = x @ self.W1 + self.b1
        h = np.maximum(0, h)
        y = h @ self.W2 + self.b2
        return y

    def forward(self, tokens, pe=None, attn_mask=None):
        x = self.E[tokens]
        if pe is not None:
            x = x + pe[None, :, :]
        x_ln = layer_norm(x)
        attn_out, A = self.self_attention(x_ln, attn_mask=attn_mask)
        x = x + attn_out
        x_ln = layer_norm(x)
        ffn_out = self.feed_forward(x_ln)
        x = x + ffn_out
        return x, A

vocab_size = 50
B, T, D = 2, 6, 32
model = EncoderLayer(vocab_size=vocab_size, d_model=D, d_ff=64, seed=0)
tokens = np.array([[1, 5, 2, 9, 9, 3], [4, 4, 7, 1, 0, 2]], dtype=np.int64)
pe = positional_encoding(T, D)
out, attn = model.forward(tokens, pe=pe)
print("out shape:", out.shape)
print("attn shape:", attn.shape)
print("attn[0] row sums:", attn[0].sum(axis=-1))

out shape: (2, 6, 32)
attn shape: (2, 6, 6)
attn[0] row sums: [1. 1. 1. 1. 1. 1.]
